# U-Net - PyTorch


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 1. Dataset - Oxford-IIIT Pet
# -----------------------------
# We use a simplified synthetic segmentation dataset for demonstration.
# Each sample is a 128x128 image with a circular or rectangular region
# that acts as the segmentation target.

class SyntheticSegDataset(Dataset):
    """Generates simple synthetic images with shape masks for segmentation."""

    def __init__(self, num_samples=1000, img_size=128):
        self.num_samples = num_samples
        self.img_size = img_size

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        img = np.zeros((3, self.img_size, self.img_size), dtype=np.float32)
        mask = np.zeros((1, self.img_size, self.img_size), dtype=np.float32)

        # Random background color
        for c in range(3):
            img[c] = np.random.uniform(0.0, 0.3)

        # Random circle
        cx = np.random.randint(30, self.img_size - 30)
        cy = np.random.randint(30, self.img_size - 30)
        r = np.random.randint(15, 35)

        Y, X = np.ogrid[:self.img_size, :self.img_size]
        circle = (X - cx) ** 2 + (Y - cy) ** 2 <= r ** 2

        # Bright circle on image
        for c in range(3):
            img[c][circle] = np.random.uniform(0.6, 1.0)

        mask[0][circle] = 1.0

        return torch.tensor(img), torch.tensor(mask)


train_dataset = SyntheticSegDataset(num_samples=1000)
test_dataset = SyntheticSegDataset(num_samples=200)

trainloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
testloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# -----------------------------
# 2. U-Net Building Blocks
# -----------------------------
class DoubleConv(nn.Module):
    """Two consecutive 3x3 convolutions, each followed by BN and ReLU."""

    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class EncoderBlock(nn.Module):
    """Downsampling block: MaxPool -> DoubleConv."""

    def __init__(self, in_channels, out_channels):
        super(EncoderBlock, self).__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        return self.conv(self.pool(x))


class DecoderBlock(nn.Module):
    """Upsampling block: TransposedConv -> Concat skip -> DoubleConv."""

    def __init__(self, in_channels, out_channels):
        super(DecoderBlock, self).__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([skip, x], dim=1)  # Skip connection (concatenation)
        return self.conv(x)


# -----------------------------
# 3. U-Net Model
# -----------------------------
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        # Encoder (contracting path)
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 256)
        self.enc4 = EncoderBlock(256, 512)

        # Bottleneck
        self.bottleneck = EncoderBlock(512, 1024)

        # Decoder (expanding path)
        self.dec4 = DecoderBlock(1024, 512)
        self.dec3 = DecoderBlock(512, 256)
        self.dec2 = DecoderBlock(256, 128)
        self.dec1 = DecoderBlock(128, 64)

        # Final 1x1 convolution
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        s1 = self.enc1(x)     # 64
        s2 = self.enc2(s1)    # 128
        s3 = self.enc3(s2)    # 256
        s4 = self.enc4(s3)    # 512

        # Bottleneck
        b = self.bottleneck(s4)  # 1024

        # Decoder with skip connections
        d4 = self.dec4(b, s4)    # 512
        d3 = self.dec3(d4, s3)   # 256
        d2 = self.dec2(d3, s2)   # 128
        d1 = self.dec1(d2, s1)   # 64

        return self.final_conv(d1)


model = UNet(in_channels=3, out_channels=1).to(device)

# -----------------------------
# 4. Loss and Optimizer
# -----------------------------
def dice_loss(pred, target, smooth=1.0):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - dice.mean()


def combined_loss(pred, target):
    bce = nn.functional.binary_cross_entropy_with_logits(pred, target)
    dl = dice_loss(pred, target)
    return bce + dl


optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_losses = []

# -----------------------------
# 5. Training Loop
# -----------------------------
for epoch in range(10):
    model.train()
    running_loss = 0.0

    for images, masks in trainloader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(trainloader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

# -----------------------------
# 6. Visualization
# -----------------------------
model.eval()
with torch.no_grad():
    sample_images, sample_masks = next(iter(testloader))
    sample_images = sample_images.to(device)
    preds = torch.sigmoid(model(sample_images)).cpu()

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i in range(4):
    # Input image
    img = sample_images[i].cpu().numpy().transpose(1, 2, 0)
    axes[0, i].imshow(np.clip(img, 0, 1))
    axes[0, i].set_title("Input")
    axes[0, i].axis("off")

    # Ground truth mask
    axes[1, i].imshow(sample_masks[i, 0], cmap="gray")
    axes[1, i].set_title("Ground Truth")
    axes[1, i].axis("off")

    # Predicted mask
    axes[2, i].imshow(preds[i, 0], cmap="gray")
    axes[2, i].set_title("Prediction")
    axes[2, i].axis("off")

plt.suptitle("U-Net Segmentation Results", fontsize=16)
plt.tight_layout()
plt.show()

# Training loss curve
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()